<a href="https://colab.research.google.com/github/a-nagar/cs4372/blob/main/Deep_Learning_Text/Hugging_Face_PyTorch_NLP_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hugging Face Transformers with PyTorch — NLP Task Demos

This notebook shows how to use pretrained Hugging Face models for common NLP tasks:

| # | Task | Approach | Model |
|---|------|----------|-------|
| 1 | Sentiment analysis | `pipeline` + under the hood | DistilBERT (SST-2), Twitter-RoBERTa |
| 2 | Emotion detection | `pipeline` (multi-class) | DistilRoBERTa-emotion |
| 3 | Named entity recognition | `pipeline` | BERT-NER |
| 4 | Zero-shot classification | `pipeline` | BART-MNLI |
| 5 | Summarization | `AutoModelForSeq2SeqLM.generate` | DistilBART-CNN |
| 6 | Translation | `AutoModelForSeq2SeqLM.generate` | MarianMT (Helsinki-NLP), NLLB |
| 7 | Extractive question answering | `AutoModelForQuestionAnswering` | DistilBERT-SQuAD |
| 8 | Fill-mask | `pipeline` | BERT |
| 9 | Sentence embeddings & similarity | `AutoModel` + mean pooling | all-MiniLM-L6-v2 |
| 10 | Text generation / instruction following | `AutoModelForCausalLM` + chat template | SmolLM2-Instruct |

**Framework note.** Transformers v5 dropped TensorFlow/Flax support and also removed the
`summarization`, `translation`, `question-answering`, and `text2text-generation` pipelines.
This notebook uses **PyTorch** throughout and does those tasks with the model classes directly,
so the code runs on both Transformers v4.4x and v5.

Runs on CPU; a GPU (e.g., Google Colab → *Runtime → Change runtime type → T4 GPU*) makes sections 5, 6 and 10 much faster.
The first run of each section downloads model weights (cached under `~/.cache/huggingface`).

## 0. Setup

In [ ]:
# Uncomment on a fresh environment (Colab already has torch + transformers)
# pip install -q torch transformers sentencepiece sacremoses

In [ ]:
import torch
import torch.nn.functional as F
import transformers
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    AutoModelForQuestionAnswering,
    AutoModelForCausalLM,
)

transformers.logging.set_verbosity_error()   # keep the output clean

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():       # Apple Silicon
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"torch {torch.__version__} | transformers {transformers.__version__} | device: {DEVICE}")

torch 2.11.0+cu130 | transformers 5.20.2 | device: cuda


## 1. Sentiment analysis

### 1a. The one-liner: `pipeline`
A `pipeline` bundles three steps: **tokenize → model forward pass → post-process** (softmax + label lookup).

In [ ]:
sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=DEVICE,
)

reviews = [
    "This course was fantastic — the projects really helped me understand transformers.",
    "The lectures were okay, but the homework instructions were confusing.",
    "I would not recommend this phone. The battery died after two days.",
]

for text, result in zip(reviews, sentiment(reviews)):
    print(f"{result['label']:<8} {result['score']:.3f}  {text}")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

POSITIVE 1.000  This course was fantastic — the projects really helped me understand transformers.
NEGATIVE 0.998  The lectures were okay, but the homework instructions were confusing.
NEGATIVE 1.000  I would not recommend this phone. The battery died after two days.


### 1b. Under the hood
The same prediction, step by step, with the tokenizer and model used directly.

In [ ]:
model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
clf_model = AutoModelForSequenceClassification.from_pretrained(model_name).to(DEVICE).eval()

# Step 1: tokenize (text -> token ids + attention mask), padded into one batch tensor
batch = tokenizer(reviews, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
print("input_ids shape:", tuple(batch["input_ids"].shape))
print("tokens of review 0:", tokenizer.convert_ids_to_tokens(batch["input_ids"][0])[:16], "...")

# Step 2: forward pass -> raw, unnormalized scores (logits)
with torch.no_grad():
    logits = clf_model(**batch).logits
print("logits:\n", logits)

# Step 3: softmax -> probabilities, argmax -> label
probs = F.softmax(logits, dim=-1)
for text, p in zip(reviews, probs):
    label = clf_model.config.id2label[p.argmax().item()]
    print(f"{label:<8} {p.max().item():.3f}  {text}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

input_ids shape: (3, 16)
tokens of review 0: ['[CLS]', 'this', 'course', 'was', 'fantastic', '—', 'the', 'projects', 'really', 'helped', 'me', 'understand', 'transformers', '.', '[SEP]', '[PAD]'] ...
logits:
 tensor([[-4.2288,  4.5498],
        [ 3.3215, -2.7690],
        [ 4.3378, -3.4549]], device='cuda:0')
POSITIVE 1.000  This course was fantastic — the projects really helped me understand transformers.
NEGATIVE 0.998  The lectures were okay, but the homework instructions were confusing.
NEGATIVE 1.000  I would not recommend this phone. The battery died after two days.


### 1c. Three-class sentiment (negative / neutral / positive)
SST-2 models only know *positive* vs *negative*. This RoBERTa model, trained on ~124M tweets, adds *neutral*.

In [ ]:
sentiment3 = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=DEVICE,
)
tweets = ["Just finished my final exam!!", "The meeting is at 3pm in room 2.410.", "My flight got cancelled again..."]
for t, r in zip(tweets, sentiment3(tweets)):
    print(f"{r['label']:<9} {r['score']:.3f}  {t}")

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

positive  0.883  Just finished my final exam!!
neutral   0.940  The meeting is at 3pm in room 2.410.
negative  0.878  My flight got cancelled again...


## 2. Emotion detection (multi-class, show all scores)
`top_k=None` returns the full probability distribution instead of only the top label.

In [ ]:
emotion = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    device=DEVICE,
)
text = "I can't believe they cancelled the concert at the last minute!"
scores = emotion(text)[0]
for s in sorted(scores, key=lambda x: -x["score"]):
    print(f"{s['label']:<9} {s['score']:.3f} " + "█" * int(s["score"] * 40))

config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  329MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  329MB            

model.safetensors: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

surprise  0.867 ██████████████████████████████████
anger     0.061 ██
sadness   0.040 █
disgust   0.012 
fear      0.010 
neutral   0.006 
joy       0.003 


## 3. Named entity recognition (NER)
Token classification: each token gets a tag (PER, ORG, LOC, MISC).
`aggregation_strategy="simple"` merges word pieces such as `Na ##gar` back into whole entities.

In [ ]:
ner = pipeline(
    "token-classification",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=DEVICE,
)
text = ("Satya Nadella announced that Microsoft will open a new AI research lab in Dallas, Texas, "
        "in partnership with the University of Texas at Dallas.")
for ent in ner(text):
    print(f"{ent['entity_group']:<5} {ent['score']:.3f}  {ent['word']}")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

PER   0.903  Satya Nadella
ORG   0.998  Microsoft
MISC  0.910  AI
LOC   0.998  Dallas
LOC   0.999  Texas
ORG   0.941  University of Texas at Dallas


## 4. Zero-shot classification
Classify text into labels the model was **never trained on**. An NLI (natural language inference) model
scores how strongly the text entails the hypothesis *"This example is about {label}."* for each label.

In [ ]:
zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=DEVICE)

text = "The Federal Reserve raised interest rates by a quarter point to fight inflation."
labels = ["economics", "sports", "politics", "technology", "health"]

result = zero_shot(text, candidate_labels=labels)
for label, score in zip(result["labels"], result["scores"]):
    print(f"{label:<11} {score:.3f}")

# multi_label=True scores each label independently (scores no longer sum to 1)
result = zero_shot(text, candidate_labels=labels, multi_label=True)
print("\nmulti-label:", {l: round(s, 3) for l, s in zip(result["labels"], result["scores"])})

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

economics   0.756
politics    0.076
technology  0.064
health      0.059
sports      0.044

multi-label: {'economics': 0.663, 'politics': 0.004, 'technology': 0.002, 'health': 0.001, 'sports': 0.0}


## 5. Summarization (abstractive)
An encoder–decoder (seq2seq) model reads the article and **generates** a new, shorter text.
`generate()` decoding settings control length and quality:

- `num_beams` — beam search keeps the k best partial summaries at each step
- `min_length` / `max_length` — summary length in tokens
- `no_repeat_ngram_size` — blocks repeated phrases

In [ ]:
sum_name = "sshleifer/distilbart-cnn-12-6"
sum_tok = AutoTokenizer.from_pretrained(sum_name)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(sum_name).to(DEVICE).eval()

def summarize(text, max_length=130, min_length=30, num_beams=4):
    inputs = sum_tok(text, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    with torch.no_grad():
        out = sum_model.generate(
            **inputs,
            num_beams=num_beams,
            max_length=max_length,
            min_length=min_length,
            length_penalty=2.0,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    return sum_tok.decode(out[0], skip_special_tokens=True)

article = """
The James Webb Space Telescope has captured the most detailed images yet of a distant galaxy cluster,
revealing thousands of galaxies, some of which formed shortly after the Big Bang. Scientists say the
infrared observatory, which launched in December 2021, can see farther back in time than any previous
telescope because it detects light that has been stretched to longer wavelengths as the universe expands.
The images show gravitational lensing, in which the mass of the foreground cluster bends and magnifies the
light from galaxies behind it. Researchers expect the data to improve estimates of when the first stars
formed and to shed light on how galaxies grew over billions of years. NASA, the European Space Agency and
the Canadian Space Agency jointly operate the telescope, which orbits the Sun about 1.5 million kilometers
from Earth.
"""

print("Original:", len(article.split()), "words")
summary = summarize(article)
print("Summary: ", len(summary.split()), "words\n")
print(summary)

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Original: 136 words
Summary:  39 words

 James Webb Space Telescope has captured the most detailed images yet of a distant galaxy cluster . Images show gravitational lensing, in which the mass of the foreground cluster bends and magnifies the light from galaxies behind it .


## 6. Translation
### 6a. MarianMT — one small model per language pair
Helsinki-NLP publishes 1,000+ `opus-mt-{src}-{tgt}` models (~300 MB each). They are fast and work well on CPU.

In [ ]:
_translators = {}   # cache loaded models

def translate(texts, src="en", tgt="fr", num_beams=4):
    name = f"Helsinki-NLP/opus-mt-{src}-{tgt}"
    if name not in _translators:
        tok = AutoTokenizer.from_pretrained(name)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(name).to(DEVICE).eval()
        _translators[name] = (tok, mdl)
    tok, mdl = _translators[name]
    batch = tok(texts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = mdl.generate(**batch, num_beams=num_beams, max_new_tokens=256)
    return tok.batch_decode(out, skip_special_tokens=True)

sentences = [
    "Machine learning models learn patterns from data.",
    "The library is closed on Sunday.",
]
for tgt in ["fr", "es", "de"]:
    print(f"--- en → {tgt}")
    for s, t in zip(sentences, translate(sentences, "en", tgt)):
        print(f"  {t}")

--- en → fr


config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  298MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

  Maschinelle Lernmodelle lernen Muster aus Daten.
  Die Bibliothek ist am Sonntag geschlossen.


### 6b. NLLB-200 — one model, 200 languages (optional, ~2.4 GB)
Set the source language on the tokenizer and force the first generated token to be the target-language code.
Language codes: [FLORES-200 list](https://github.com/facebookresearch/flores/blob/main/flores200/README.md#languages-in-flores-200) (e.g., `hin_Deva`, `zho_Hans`, `arb_Arab`).

In [ ]:
RUN_NLLB = False   # set True to download and run (~2.4 GB)

if RUN_NLLB:
    nllb_name = "facebook/nllb-200-distilled-600M"
    nllb_tok = AutoTokenizer.from_pretrained(nllb_name, src_lang="eng_Latn")
    nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_name).to(DEVICE).eval()

    text = "Machine learning models learn patterns from data."
    inputs = nllb_tok(text, return_tensors="pt").to(DEVICE)
    for lang in ["hin_Deva", "zho_Hans", "arb_Arab", "jpn_Jpan"]:
        with torch.no_grad():
            out = nllb_model.generate(
                **inputs,
                forced_bos_token_id=nllb_tok.convert_tokens_to_ids(lang),
                max_new_tokens=100,
            )
        print(f"{lang}: {nllb_tok.decode(out[0], skip_special_tokens=True)}")

## 7. Extractive question answering
The model doesn't generate an answer; it **points to a span** of the context.
It outputs two scores per token: how likely the token is the answer's **start**, and how likely it is the **end**.
We pick the best valid (start ≤ end, inside the context) span.

In [ ]:
qa_name = "distilbert/distilbert-base-cased-distilled-squad"
qa_tok = AutoTokenizer.from_pretrained(qa_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_name).to(DEVICE).eval()

def answer_question(question, context, max_answer_len=30):
    inputs = qa_tok(question, context, return_tensors="pt", truncation="only_second", max_length=384)
    seq_ids = inputs.sequence_ids(0)                      # None=special, 0=question, 1=context
    with torch.no_grad():
        out = qa_model(**inputs.to(DEVICE))
    start_logits, end_logits = out.start_logits[0], out.end_logits[0]

    # score every (start, end) pair = start_logit + end_logit
    scores = start_logits[:, None] + end_logits[None, :]
    n = scores.size(0)
    is_ctx = torch.tensor([s == 1 for s in seq_ids], device=scores.device)
    valid = torch.ones(n, n, dtype=torch.bool, device=scores.device).triu()           # start <= end
    valid &= ~torch.ones(n, n, dtype=torch.bool, device=scores.device).triu(max_answer_len)  # length limit
    valid &= is_ctx[:, None] & is_ctx[None, :]                                        # inside context
    scores = scores.masked_fill(~valid, float("-inf"))

    best = scores.argmax()
    start, end = divmod(best.item(), n)
    answer = qa_tok.decode(inputs["input_ids"][0][start:end + 1])
    # confidence: softmax over all valid spans
    confidence = F.softmax(scores.flatten(), dim=0)[best].item()
    return answer, confidence

context = """
The University of Texas at Dallas was founded in 1969 and is located in Richardson, Texas.
The Erik Jonsson School of Engineering and Computer Science offers degrees in computer science,
electrical engineering, and software engineering. The university's mascot is Temoc, a flame.
"""
for q in ["When was UT Dallas founded?",
          "Where is the university located?",
          "What is the mascot?",
          "Which degrees does the engineering school offer?"]:
    ans, conf = answer_question(q, context)
    print(f"Q: {q}\nA: {ans}   (confidence {conf:.2f})\n")

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Q: When was UT Dallas founded?
A: 1969   (confidence 0.99)

Q: Where is the university located?
A: Richardson, Texas   (confidence 0.90)

Q: What is the mascot?
A: Temoc   (confidence 0.91)

Q: Which degrees does the engineering school offer?
A: computer science, electrical engineering, and software engineering   (confidence 0.92)



## 8. Fill-mask (masked language modeling)
This is BERT's pre-training task: predict the hidden word from context on both sides.
It is also a quick way to show biases a model has learned from its training data.

In [ ]:
fill = pipeline("fill-mask", model="google-bert/bert-base-uncased", device=DEVICE)
mask = fill.tokenizer.mask_token          # "[MASK]" for BERT, "<mask>" for RoBERTa

for sentence in [f"Paris is the {mask} of France.",
                 f"The doctor said that {mask} would be back soon.",
                 f"The nurse said that {mask} would be back soon."]:
    preds = fill(sentence, top_k=5)
    print(sentence)
    print("   ", ", ".join(f"{p['token_str']} ({p['score']:.2f})" for p in preds))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Paris is the [MASK] of France.
    capital (1.00), heart (0.00), center (0.00), centre (0.00), city (0.00)
The doctor said that [MASK] would be back soon.
    he (0.42), she (0.22), they (0.13), i (0.02), we (0.01)
The nurse said that [MASK] would be back soon.
    she (0.42), he (0.21), they (0.14), i (0.02), we (0.01)


## 9. Sentence embeddings & semantic similarity
Encode each sentence as a vector (mean of token embeddings, ignoring padding), then compare with cosine similarity.
This is the building block of semantic search, clustering, and retrieval-augmented generation (RAG).

In [ ]:
emb_name = "sentence-transformers/all-MiniLM-L6-v2"
emb_tok = AutoTokenizer.from_pretrained(emb_name)
emb_model = AutoModel.from_pretrained(emb_name).to(DEVICE).eval()

def embed(texts):
    batch = emb_tok(texts, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        token_emb = emb_model(**batch).last_hidden_state            # (batch, seq_len, 384)
    mask = batch["attention_mask"].unsqueeze(-1).float()             # (batch, seq_len, 1)
    sent_emb = (token_emb * mask).sum(1) / mask.sum(1).clamp(min=1e-9)  # mean pooling
    return F.normalize(sent_emb, p=2, dim=1)                          # unit length

docs = [
    "How do I reset my password?",
    "Steps to recover a forgotten login credential",
    "The stock market fell sharply today.",
    "Equities dropped as investors worried about inflation.",
    "My favorite food is pizza.",
]
E = embed(docs)
sim = E @ E.T                                                         # cosine similarity matrix

print("Similarity matrix:")
for i, d in enumerate(docs):
    print(" ".join(f"{v:5.2f}" for v in sim[i].tolist()), " ", d[:45])

# Semantic search: best match for a query
query = "I can't log into my account"
scores = (embed([query]) @ E.T)[0]
best = scores.argmax().item()
print(f"\nQuery: {query!r}\nBest match: {docs[best]!r} (score {scores[best]:.2f})")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Similarity matrix:
 1.00  0.57  0.03 -0.10  0.07   How do I reset my password?
 0.57  1.00  0.01  0.01  0.05   Steps to recover a forgotten login credential
 0.03  0.01  1.00  0.48 -0.03   The stock market fell sharply today.
-0.10  0.01  0.48  1.00 -0.07   Equities dropped as investors worried about i
 0.07  0.05 -0.03 -0.07  1.00   My favorite food is pizza.

Query: "I can't log into my account"
Best match: 'How do I reset my password?' (score 0.58)


## 10. Text generation with a small instruction-tuned LLM
Decoder-only LLMs can do *all* of the tasks above from a prompt. The **chat template** converts a list of
messages into the exact prompt format the model was fine-tuned on.

`SmolLM2-360M-Instruct` is tiny (~720 MB) so it runs on CPU; quality is far below large models, but the code
is identical for bigger ones (e.g., `Qwen/Qwen2.5-1.5B-Instruct`, `meta-llama/Llama-3.2-3B-Instruct` — the latter needs a Hugging Face login).

In [ ]:
llm_name = "HuggingFaceTB/SmolLM2-360M-Instruct"
llm_tok = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(llm_name).to(DEVICE).eval()

def chat(user_msg, system="You are a helpful assistant.", max_new_tokens=200,
         do_sample=True, temperature=0.7, top_p=0.9):
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": user_msg}]
    prompt = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tok(prompt, return_tensors="pt").to(DEVICE)
    gen_kwargs = dict(max_new_tokens=max_new_tokens, do_sample=do_sample,
                      pad_token_id=llm_tok.eos_token_id)
    if do_sample:
        gen_kwargs.update(temperature=temperature, top_p=top_p)
    with torch.no_grad():
        out = llm.generate(**inputs, **gen_kwargs)
    new_tokens = out[0, inputs["input_ids"].shape[1]:]      # drop the prompt
    return llm_tok.decode(new_tokens, skip_special_tokens=True).strip()

# Peek at what the chat template produces
print(llm_tok.apply_chat_template([{"role": "user", "content": "Hi!"}], tokenize=False, add_generation_prompt=True))

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  724MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant



In [ ]:
print(chat("Explain what a transformer neural network is in three sentences for a college sophomore."))

A transformer neural network is a type of artificial neural network that is particularly effective at understanding and processing the intricacies of human language. It is designed to learn patterns in text data, such as sentences, paragraphs, and even entire books, allowing it to recognize the relationships between words, phrases, and even entire ideas. This enables the network to generate coherent and meaningful responses, much like how a human would.


In [ ]:
# The same LLM doing earlier tasks via prompting (greedy decoding for repeatability)
print("Sentiment:", chat("Classify the sentiment as positive, negative, or neutral. Reply with one word.\n\n"
                         "Text: The homework instructions were confusing.", do_sample=False, max_new_tokens=5))
print("\nSummary:", chat(f"Summarize in one sentence:\n{article}", do_sample=False, max_new_tokens=60))

Sentiment: Negative

Summary: The James Webb Space Telescope has captured the most detailed images of a distant galaxy cluster, revealing thousands of galaxies and providing insights into the formation of the universe.


## Decoding strategies at a glance

| Strategy | `generate()` settings | Behavior |
|---|---|---|
| Greedy | `do_sample=False` | Always picks the most likely token; deterministic, can be repetitive |
| Beam search | `num_beams=4` | Keeps several candidate sequences; good for summarization/translation |
| Sampling | `do_sample=True, temperature=0.7` | Random draws; higher temperature = more creative |
| Nucleus (top-p) | `do_sample=True, top_p=0.9` | Samples only from the smallest token set covering 90% of the probability |

## Exercises
1. Swap the sentiment model for `nlptown/bert-base-multilingual-uncased-sentiment` (1–5 stars, multilingual) and test reviews in Spanish or French.
2. Translate a sentence en → fr → en ("back-translation") and compare it to the original.
3. Use the embeddings from Section 9 to build a mini semantic-search engine over 20 FAQ entries.
4. Compare the extractive QA model (Section 7) with the LLM (Section 10) on the same questions. When does each fail?
5. Run Section 10 with `temperature` = 0.2, 0.7, and 1.5. Describe how the output changes.